# Stage 1 – Data Summary (Baseline vs 1‑cycle Scenarios)

This notebook produces **table‑based** summaries for all output CSVs.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display


In [ ]:
# --- User configuration -----------------------------------------------------
PROJECT_DIR = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN_NAME = "low_RE_mod_elec_iter0"
CLIMATE_SCENARIO = "historical_1980_2019"

RUNS = {
    "baseline": "stagel_3month_base",
    "pcap2160_1cycle": "stagel_3month_1cycle",
    "pcap720_1cycle": "stagel_1month_1cycle",
}

YEAR = 1985

output_root = PROJECT_DIR / "runs" / RUN_NAME / "outputs" / CLIMATE_SCENARIO
print("Runs:", RUNS)
print("Year:", YEAR)


In [ ]:
def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def tidy_bus_df(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def total_ts(df: pd.DataFrame, value_name: str) -> pd.Series:
    return df.groupby("timestamp")[value_name].sum()


def load_seasonal_soc(run_dir: Path, year: int) -> pd.Series:
    df = pd.read_csv(run_dir / f"storage_state_seasonal_{year}.csv")
    if df.empty:
        return pd.Series(dtype=float)
    cols = [c for c in df.columns if c not in ("zone",)]
    df = df[cols]
    if not df.empty and df.iloc[0, 0] == "bus_id":
        df = df.iloc[1:]
    value_cols = [c for c in df.columns if c != "bus_id"]
    df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")
    soc = df[value_cols].sum(axis=0)
    soc.index = pd.to_datetime(soc.index, errors="coerce")
    soc = soc.dropna()
    if hasattr(soc.index, 'tz') and soc.index.tz is not None:
        soc.index = soc.index.tz_convert(None)
    return soc


In [ ]:
# --- Load all runs ----------------------------------------------------------
run_data = {}
for label, name in RUNS.items():
    run_dir = output_root / name
    run_data[label] = {
        "charge_base": tidy_storage_df(pd.read_csv(run_dir / f"charge_base_{YEAR}.csv"), "charge"),
        "discharge_base": tidy_storage_df(pd.read_csv(run_dir / f"discharge_base_{YEAR}.csv"), "discharge"),
        "charge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"charge_seasonal_{YEAR}.csv"), "charge"),
        "discharge_seasonal": tidy_storage_df(pd.read_csv(run_dir / f"discharge_seasonal_{YEAR}.csv"), "discharge"),
        "load_shed": tidy_bus_df(run_dir / f"load_shedding_{YEAR}.csv", "load_shedding"),
        "wind_curt": tidy_bus_df(run_dir / f"wind_curtailment_{YEAR}.csv", "wind_curtailment"),
        "solar_curt": tidy_bus_df(run_dir / f"solar_curtailment_{YEAR}.csv", "solar_curtailment"),
        "soc_seasonal": load_seasonal_soc(run_dir, YEAR),
    }

print("Loaded:", list(run_data.keys()))


## A) Energy totals (MWh)

In [ ]:
def safe_sum(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0).sum()

rows = []
for label, data in run_data.items():
    base_dis = total_ts(data["discharge_base"], "discharge")
    seas_dis = total_ts(data["discharge_seasonal"], "discharge")
    base_ch = total_ts(data["charge_base"], "charge")
    seas_ch = total_ts(data["charge_seasonal"], "charge")
    ls = total_ts(data["load_shed"], "load_shedding")
    wind = total_ts(data["wind_curt"], "wind_curtailment")
    solar = total_ts(data["solar_curt"], "solar_curtailment")

    rows.append({
        "run": label,
        "base_charge_MWh": safe_sum(base_ch),
        "base_discharge_MWh": safe_sum(base_dis),
        "seasonal_charge_MWh": safe_sum(seas_ch),
        "seasonal_discharge_MWh": safe_sum(seas_dis),
        "load_shed_MWh": safe_sum(ls),
        "wind_curt_MWh": safe_sum(wind),
        "solar_curt_MWh": safe_sum(solar),
    })

summary = pd.DataFrame(rows)
for c in summary.columns[1:]:
    summary[c] = summary[c].round(2)

display(summary)


## B) Deltas vs baseline

In [ ]:
base = summary[summary["run"] == "baseline"].set_index("run")
rows = []
for label in summary["run"]:
    if label == "baseline":
        continue
    row = summary[summary["run"] == label].set_index("run")
    delta = row - base
    delta["run"] = label
    rows.append(delta.reset_index(drop=True))

delta_df = pd.concat(rows, ignore_index=True)
cols = ["run"] + [c for c in delta_df.columns if c != "run"]
display(delta_df[cols])


## C) Load shedding by bus (top 15)

In [ ]:
for label, data in run_data.items():
    ls = data["load_shed"]
    by_bus = ls.groupby("bus_id", as_index=False)["load_shedding"].sum()
    print(f"
{label} – top 15 shedding buses")
    display(by_bus.sort_values("load_shedding", ascending=False).head(15))


## D) Seasonal SOC stats

In [ ]:
rows = []
for label, data in run_data.items():
    soc = data["soc_seasonal"]
    if soc.empty:
        rows.append({"run": label, "soc_max_MWh": float("nan"), "soc_min_MWh": float("nan"), "soc_end_MWh": float("nan")})
        continue
    rows.append({
        "run": label,
        "soc_max_MWh": float(soc.max()),
        "soc_min_MWh": float(soc.min()),
        "soc_end_MWh": float(soc.iloc[-1]),
    })

soc_df = pd.DataFrame(rows)
for c in soc_df.columns[1:]:
    soc_df[c] = soc_df[c].round(2)

display(soc_df)


## E) Seasonal cycle usage (using installed energy)

In [ ]:
rows = []
for label, name in RUNS.items():
    run_dir = output_root / name
    debug = pd.read_csv(run_dir / "_storage_debug.csv")
    seasonal = debug[debug["is_seasonal"] == 1]
    total_energy = seasonal["storage_capacity_mwh"].sum()

    seas_dis = total_ts(run_data[label]["discharge_seasonal"], "discharge").sum()
    cycles = seas_dis / total_energy if total_energy > 0 else 0.0

    rows.append({
        "run": label,
        "total_energy_MWh": total_energy,
        "seasonal_discharge_MWh": seas_dis,
        "approx_cycles": cycles,
    })

display(pd.DataFrame(rows))


## E) Seasonal month‑level diagnostics (net energy + SOC)
Checks whether seasonal storage is shifting energy across months rather than daily cycling.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def _strip_tz(idx):
    if hasattr(idx, 'tz') and idx.tz is not None:
        return idx.tz_convert(None)
    return idx

def monthly_sum(series: pd.Series) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    s = series.copy()
    s.index = _strip_tz(s.index)
    return s.resample("ME").sum()

def monthly_mean(series: pd.Series) -> pd.Series:
    if series is None or len(series) == 0:
        return pd.Series(dtype=float)
    s = series.copy()
    s.index = _strip_tz(s.index)
    return s.resample("ME").mean()

# --- Monthly net seasonal energy (charge - discharge)
monthly_net = {}
monthly_dis = {}
monthly_ls = {}
for label, data in run_data.items():
    seas_ch = total_ts(data["charge_seasonal"], "charge")
    seas_dis = total_ts(data["discharge_seasonal"], "discharge")
    net = seas_ch - seas_dis
    monthly_net[label] = monthly_sum(net)
    monthly_dis[label] = monthly_sum(seas_dis)
    monthly_ls[label] = monthly_sum(total_ts(data["load_shed"], "load_shedding"))

monthly_net_df = pd.DataFrame(monthly_net).sort_index()
monthly_dis_df = pd.DataFrame(monthly_dis).sort_index()
monthly_ls_df = pd.DataFrame(monthly_ls).sort_index()

print("Monthly net seasonal energy (MWh):")
display(monthly_net_df.round(1))

plot_net = monthly_net_df.dropna(how="all", axis=1)
if not plot_net.empty:
    ax = plot_net.plot(figsize=(12,4), marker='o', title="Monthly net seasonal energy (charge − discharge)")
    ax.set_ylabel("MWh")
    plt.tight_layout()
    plt.show()

# --- Monthly SOC mean (seasonal)
monthly_soc = {}
for label, data in run_data.items():
    soc = data["soc_seasonal"]
    monthly_soc[label] = monthly_mean(soc)

monthly_soc_df = pd.DataFrame(monthly_soc).sort_index()
print("Monthly mean seasonal SOC (MWh):")
display(monthly_soc_df.round(1))

plot_soc = monthly_soc_df.dropna(how="all", axis=1)
if not plot_soc.empty:
    ax = plot_soc.plot(figsize=(12,4), marker='o', title="Monthly mean seasonal SOC")
    ax.set_ylabel("MWh")
    plt.tight_layout()
    plt.show()

# --- Optional: monthly discharge vs load shedding alignment
align = pd.DataFrame({
    "seasonal_discharge_MWh": monthly_dis_df.sum(axis=1),
    "load_shed_MWh": monthly_ls_df.sum(axis=1),
})
print("Monthly seasonal discharge vs load shedding (all runs summed):")
display(align.round(1))